In [1]:
# ═══════════════════════════════════════════════════════════
# NOTEBOOK 03 — MODEL EVALUATION
# Marathi Mitra — Testing the fine-tuned model
#
# This notebook:
# → Loads fine-tuned model FROM Hugging Face Hub
# → Tests on SEEN words (training set)
# → Tests on UNSEEN words (generalisation test)
# → Generates before/after comparison
# → Produces output ready for README
#
# Requires: GPU (T4 minimum)
# ═══════════════════════════════════════════════════════════

print("🔍 Marathi Mitra — Model Evaluation")
print("=" * 50)
print()
print("Key question this notebook answers:")
print("→ Did the model MEMORIZE or actually LEARN?")
print()
print("If memorized: good on seen words, fails on unseen")
print("If learned:   good on BOTH seen and unseen words")

🔍 Marathi Mitra — Model Evaluation

Key question this notebook answers:
→ Did the model MEMORIZE or actually LEARN?

If memorized: good on seen words, fails on unseen
If learned:   good on BOTH seen and unseen words


In [ ]:
!pip install -q "numpy>=2.0"
!pip install -q transformers==4.44.0
!pip install -q peft==0.12.0
!pip install -q bitsandbytes==0.45.3
!pip install -q accelerate==0.33.0
!pip install -q huggingface_hub

print("✅ Libraries installed!")
print("⚠️  Runtime → Restart Session → skip this cell → run Cell 2")

In [ ]:
import transformers
import peft
import bitsandbytes
import accelerate
import huggingface_hub

print(f"transformers:    {transformers.__version__}")
print(f"peft:            {peft.__version__}")
print(f"bitsandbytes:    {bitsandbytes.__version__}")
print(f"accelerate:      {accelerate.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
print("\n✅ All imports successful!")

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN    = userdata.get("HF_TOKEN")
HF_USERNAME = "ninadp"

REPOS = {
    "base": "microsoft/Phi-3-mini-4k-instruct",
    "v1":   f"{HF_USERNAME}/marathi-mitra-phi3",
    "v2":   f"{HF_USERNAME}/marathi-mitra-phi3-v2",
}

login(token=HF_TOKEN)
print(f"✅ Logged in as: {HF_USERNAME}")
print(f"\nModels to evaluate:")
for name, repo in REPOS.items():
    print(f"  {name}: {repo}")

In [ ]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ── Test words ────────────────────────────────────────────
SEEN_WORDS = {
    "butterfly": {"marathi": "फुलपाखरू", "pronunciation": "Phul-pakh-roo"},
    "mother":    {"marathi": "आई",        "pronunciation": "Aa-ee"},
    "rain":      {"marathi": "पाऊस",      "pronunciation": "Paa-oos"},
    "elephant":  {"marathi": "हत्ती",     "pronunciation": "Hat-tee"},
    "school":    {"marathi": "शाळा",      "pronunciation": "Shaa-la"},
}

UNSEEN_WORDS = {
    "apple":  {"marathi": "सफरचंद", "pronunciation": "Sa-far-chand"},
    "star":   {"marathi": "तारा",    "pronunciation": "Taa-raa"},
    "tiger":  {"marathi": "वाघ",     "pronunciation": "Vaagh"},
    "ocean":  {"marathi": "समुद्र",  "pronunciation": "Sa-mu-dra"},
    "dance":  {"marathi": "नृत्य",   "pronunciation": "Nru-tya"},
}

# ── Results store ─────────────────────────────────────────
ALL_EVAL_RESULTS = {}


# ── Load model ────────────────────────────────────────────
def load_model(version):
    """Load base, v1 or v2 model."""
    BASE = "microsoft/Phi-3-mini-4k-instruct"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    print(f"Loading base model...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="eager",
    )

    if version == "base":
        print("✅ Using base model (no adapter)")
        model = base
    else:
        adapter_repo = REPOS[version]
        print(f"Loading {version} adapter from {adapter_repo}...")
        model = PeftModel.from_pretrained(
            base, adapter_repo, token=HF_TOKEN
        )

    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(
        BASE, trust_remote_code=True
    )
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"✅ {version} model ready!")
    return model, tokenizer


# ── Generate response ─────────────────────────────────────
def generate(word, model, tokenizer):
    prompt = f"""### Instruction:
You are Marathi Mitra, a friendly Marathi teacher for kids. \
When given an English word, teach it in Marathi with the word \
in Devanagari script, pronunciation, a simple story sentence, \
and a fun fact. Always be encouraging and kid-friendly.

### Input:
Teach me the Marathi word for: {word}

### Response:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text.split("### Response:")[-1].strip()


# ── Extract fields ────────────────────────────────────────
def extract_fields(output):
    fields = {}

    marathi_match = re.search(r"is \*\*([^*]+)\*\*", output)
    fields["marathi_word"] = (
        marathi_match.group(1).strip() if marathi_match else ""
    )

    pronun_match = re.search(r"How to say it:\*?\*?\s*(.+)", output)
    fields["pronunciation"] = (
        pronun_match.group(1).strip() if pronun_match else ""
    )

    return fields


# ── Score one output ──────────────────────────────────────
def evaluate_output(output, word, expected_marathi, expected_pronun):
    fields = extract_fields(output)

    # Criteria 1 — Field presence (40%)
    presence = {
        "Has Devanagari":    bool(re.search(r"[\u0900-\u097F]", output)),
        "Has pronunciation": "How to say" in output,
        "Has sentence":      "Example sentence" in output,
        "Has fun fact":      "Fun Fact" in output,
        "Has emojis":        "🌟" in output and "📢" in output,
    }
    presence_score = sum(presence.values()) / len(presence) * 100

    # Criteria 2 — Exact match (60%)
    exact = {}
    exact["Marathi word correct"] = (
        expected_marathi in fields["marathi_word"] or
        expected_marathi in output
    )
    m = fields["pronunciation"].lower().replace(" ", "").replace("-", "")
    g = expected_pronun.lower().replace(" ", "").replace("-", "")
    exact["Pronunciation correct"] = m == g
    exact_score = sum(exact.values()) / len(exact) * 100

    combined = presence_score * 0.40 + exact_score * 0.60
    return combined, presence, exact, fields


# ── Evaluate one model on word set ────────────────────────
def evaluate_model(model, tokenizer, version, word_dict, set_name):
    print(f"\n{'═'*60}")
    print(f"  {version.upper()} — {set_name}")
    print(f"{'═'*60}")

    outputs = {}
    scores  = {}
    total   = 0

    for word, golden in word_dict.items():
        print(f"\n  📝 {word.upper()}")

        output = generate(word, model, tokenizer)
        outputs[word] = output
        print(f"  🤖 {output[:200]}{'...' if len(output)>200 else ''}")

        score, presence, exact, fields = evaluate_output(
            output, word,
            golden["marathi"],
            golden["pronunciation"],
        )
        scores[word] = score
        total += score

        print(f"\n  📊 Criteria 1 — Field Presence:")
        for k, v in presence.items():
            print(f"     {'✅' if v else '❌'} {k}")

        print(f"\n  📊 Criteria 2 — Exact Match:")
        print(f"     Expected: {golden['marathi']} / {golden['pronunciation']}")
        print(f"     Model:    {fields['marathi_word'] or 'not found'} / {fields['pronunciation'] or 'not found'}")
        for k, v in exact.items():
            print(f"     {'✅' if v else '❌'} {k}")

        print(f"\n  Score: {score:.1f}%")

    avg = total / len(word_dict)
    print(f"\n  {'─'*50}")
    print(f"  {set_name} Average: {avg:.1f}%")

    return outputs, scores, avg


print("✅ Utilities loaded!")
print(f"   Seen words:   {list(SEEN_WORDS.keys())}")
print(f"   Unseen words: {list(UNSEEN_WORDS.keys())}")

In [ ]:
# ── Load and evaluate base model ──────────────────────────
print("Loading BASE model...")
base_model, tokenizer = load_model("base")

# Seen words
base_seen_out, base_seen_scores, base_seen_avg = evaluate_model(
    base_model, tokenizer, "base", SEEN_WORDS, "SEEN WORDS"
)

# Unseen words
base_unseen_out, base_unseen_scores, base_unseen_avg = evaluate_model(
    base_model, tokenizer, "base", UNSEEN_WORDS, "UNSEEN WORDS"
)

ALL_EVAL_RESULTS["base"] = {
    "seen_avg":    base_seen_avg,
    "unseen_avg":  base_unseen_avg,
    "seen_scores": base_seen_scores,
    "unseen_scores": base_unseen_scores,
}

print(f"\n✅ Base model evaluation complete!")
print(f"   Seen:   {base_seen_avg:.1f}%")
print(f"   Unseen: {base_unseen_avg:.1f}%")

# Free memory
del base_model
torch.cuda.empty_cache()
print("✅ Memory freed")

In [ ]:
# ── Load and evaluate v1 model ────────────────────────────
print("Loading v1 model (30 examples)...")
v1_model, tokenizer = load_model("v1")

# Seen words
v1_seen_out, v1_seen_scores, v1_seen_avg = evaluate_model(
    v1_model, tokenizer, "v1", SEEN_WORDS, "SEEN WORDS"
)

# Unseen words
v1_unseen_out, v1_unseen_scores, v1_unseen_avg = evaluate_model(
    v1_model, tokenizer, "v1", UNSEEN_WORDS, "UNSEEN WORDS"
)

ALL_EVAL_RESULTS["v1"] = {
    "seen_avg":      v1_seen_avg,
    "unseen_avg":    v1_unseen_avg,
    "seen_scores":   v1_seen_scores,
    "unseen_scores": v1_unseen_scores,
}

print(f"\n✅ v1 evaluation complete!")
print(f"   Seen:   {v1_seen_avg:.1f}%")
print(f"   Unseen: {v1_unseen_avg:.1f}%")

# Free memory
del v1_model
torch.cuda.empty_cache()
print("✅ Memory freed")

In [ ]:
# ── Load and evaluate v2 model ────────────────────────────
print("Loading v2 model (250 examples)...")
v2_model, tokenizer = load_model("v2")

# Seen words
v2_seen_out, v2_seen_scores, v2_seen_avg = evaluate_model(
    v2_model, tokenizer, "v2", SEEN_WORDS, "SEEN WORDS"
)

# Unseen words
v2_unseen_out, v2_unseen_scores, v2_unseen_avg = evaluate_model(
    v2_model, tokenizer, "v2", UNSEEN_WORDS, "UNSEEN WORDS"
)

ALL_EVAL_RESULTS["v2"] = {
    "seen_avg":      v2_seen_avg,
    "unseen_avg":    v2_unseen_avg,
    "seen_scores":   v2_seen_scores,
    "unseen_scores": v2_unseen_scores,
}

print(f"\n✅ v2 evaluation complete!")
print(f"   Seen:   {v2_seen_avg:.1f}%")
print(f"   Unseen: {v2_unseen_avg:.1f}%")

# Free memory
del v2_model
torch.cuda.empty_cache()
print("✅ Memory freed")

In [ ]:
# ── Complete results summary ──────────────────────────────
print("\n" + "═"*70)
print("COMPLETE EVALUATION SUMMARY")
print("═"*70)

# ── Model comparison table ────────────────────────────────
print(f"\n{'Model':<10} {'Training':<12} {'Seen':>8} {'Unseen':>8} {'Overall':>8} {'Gap':>8}")
print("─"*60)

for version, label, examples in [
    ("base", "Base Phi-3",  "0"),
    ("v1",   "Fine-tuned", "30"),
    ("v2",   "Fine-tuned", "250"),
]:
    r       = ALL_EVAL_RESULTS[version]
    seen    = r["seen_avg"]
    unseen  = r["unseen_avg"]
    overall = (seen + unseen) / 2
    gap     = seen - unseen

    print(
        f"{version:<10} "
        f"{examples+' examples':<12} "
        f"{seen:>7.1f}% "
        f"{unseen:>7.1f}% "
        f"{overall:>7.1f}% "
        f"{gap:>7.1f}%"
    )

print("═"*60)

# ── Improvement analysis ──────────────────────────────────
print("\n📈 IMPROVEMENT ANALYSIS")
print("─"*60)

base_overall = (ALL_EVAL_RESULTS["base"]["seen_avg"] +
                ALL_EVAL_RESULTS["base"]["unseen_avg"]) / 2
v1_overall   = (ALL_EVAL_RESULTS["v1"]["seen_avg"] +
                ALL_EVAL_RESULTS["v1"]["unseen_avg"]) / 2
v2_overall   = (ALL_EVAL_RESULTS["v2"]["seen_avg"] +
                ALL_EVAL_RESULTS["v2"]["unseen_avg"]) / 2

print(f"Base → v1:  {v1_overall - base_overall:+.1f}% (30 examples)")
print(f"v1   → v2:  {v2_overall - v1_overall:+.1f}% (250 examples)")
print(f"Base → v2:  {v2_overall - base_overall:+.1f}% (total improvement)")

# ── Per word breakdown ────────────────────────────────────
print("\n📊 PER WORD BREAKDOWN — SEEN WORDS")
print("─"*60)
print(f"{'Word':<15} {'Base':>8} {'v1':>8} {'v2':>8} {'Trend':>8}")
print("─"*50)

for word in SEEN_WORDS:
    base_s = ALL_EVAL_RESULTS["base"]["seen_scores"].get(word, 0)
    v1_s   = ALL_EVAL_RESULTS["v1"]["seen_scores"].get(word, 0)
    v2_s   = ALL_EVAL_RESULTS["v2"]["seen_scores"].get(word, 0)
    trend  = "📈" if v2_s > v1_s > base_s else "📊" if v2_s > base_s else "📉"
    print(f"{word:<15} {base_s:>7.1f}% {v1_s:>7.1f}% {v2_s:>7.1f}% {trend:>6}")

print("\n📊 PER WORD BREAKDOWN — UNSEEN WORDS")
print("─"*60)
print(f"{'Word':<15} {'Base':>8} {'v1':>8} {'v2':>8} {'Trend':>8}")
print("─"*50)

for word in UNSEEN_WORDS:
    base_s = ALL_EVAL_RESULTS["base"]["unseen_scores"].get(word, 0)
    v1_s   = ALL_EVAL_RESULTS["v1"]["unseen_scores"].get(word, 0)
    v2_s   = ALL_EVAL_RESULTS["v2"]["unseen_scores"].get(word, 0)
    trend  = "📈" if v2_s > v1_s > base_s else "📊" if v2_s > base_s else "📉"
    print(f"{word:<15} {base_s:>7.1f}% {v1_s:>7.1f}% {v2_s:>7.1f}% {trend:>6}")

# ── Generalisation analysis ───────────────────────────────
print("\n📊 GENERALISATION ANALYSIS")
print("─"*60)
print("(Gap = Seen - Unseen, lower gap = better generalisation)")
print()

for version in ["base", "v1", "v2"]:
    r   = ALL_EVAL_RESULTS[version]
    gap = r["seen_avg"] - r["unseen_avg"]
    if gap < 10:
        verdict = "✅ Excellent"
    elif gap < 25:
        verdict = "✅ Good"
    elif gap < 40:
        verdict = "⚠️  Moderate"
    else:
        verdict = "❌ Poor (overfitting)"
    print(f"  {version}: gap={gap:.1f}% → {verdict}")

# ── Visual bar chart ──────────────────────────────────────
print("\n📊 VISUAL COMPARISON")
print("─"*60)

for version, label in [("base","Base"),("v1","v1"),("v2","v2")]:
    r    = ALL_EVAL_RESULTS[version]
    seen = r["seen_avg"]
    uns  = r["unseen_avg"]
    s_bar = "█" * int(seen/10) + "░" * (10-int(seen/10))
    u_bar = "█" * int(uns/10)  + "░" * (10-int(uns/10))
    print(f"\n  {label} Seen:   {s_bar} {seen:.1f}%")
    print(f"  {label} Unseen: {u_bar} {uns:.1f}%")

# ── README content ────────────────────────────────────────
print("\n" + "═"*70)
print("README CONTENT — Copy this into your README.md")
print("═"*70)

b_seen   = ALL_EVAL_RESULTS["base"]["seen_avg"]
b_unseen = ALL_EVAL_RESULTS["base"]["unseen_avg"]
v1_seen  = ALL_EVAL_RESULTS["v1"]["seen_avg"]
v1_uns   = ALL_EVAL_RESULTS["v1"]["unseen_avg"]
v2_seen  = ALL_EVAL_RESULTS["v2"]["seen_avg"]
v2_uns   = ALL_EVAL_RESULTS["v2"]["unseen_avg"]

print(f"""
## Evaluation Results

| Model | Training Data | Seen Words | Unseen Words | Overall |
|-------|--------------|------------|--------------|---------|
| Base Phi-3 | 0 examples | {b_seen:.1f}% | {b_unseen:.1f}% | {(b_seen+b_unseen)/2:.1f}% |
| v1 (Exp4) | 30 examples | {v1_seen:.1f}% | {v1_uns:.1f}% | {(v1_seen+v1_uns)/2:.1f}% |
| v2 (Exp4) | 250 examples | {v2_seen:.1f}% | {v2_uns:.1f}% | {(v2_seen+v2_uns)/2:.1f}% |

**Evaluation criteria:**
- Field presence (40%) — all sections present in output
- Exact match (60%) — correct Marathi word + pronunciation

**Seen words tested:** {', '.join(SEEN_WORDS.keys())}
**Unseen words tested:** {', '.join(UNSEEN_WORDS.keys())}
""")